In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
!pip install -q kaggle



In [6]:
!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection

Dataset URL: https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection
License(s): copyright-authors


In [7]:
!unzip brain-mri-images-for-brain-tumor-detection.zip -d ./MRI_BRAIN

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
import os
import glob
import numpy as np
import cv2
import tensorflow as tf

# Function to preprocess MRI images
def preprocess_mri_jpg(file_path, target_shape=(128, 128, 64)):
    file_path = file_path.numpy().decode('utf-8')
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (target_shape[0], target_shape[1]))
    img = img.astype(np.float32) / 255.0
    volume = np.stack([img] * target_shape[2], axis=-1)  # Create 3D volume
    volume = np.expand_dims(volume, axis=-1)  # Add channel dimension
    return volume

# Function to create a TensorFlow dataset
def create_dataset(tumor_dir, no_tumor_dir, batch_size=8):
    tumor_files = glob.glob(os.path.join(tumor_dir, "*.jpg"))
    no_tumor_files = glob.glob(os.path.join(no_tumor_dir, "*.jpg"))

    file_paths = np.array(tumor_files + no_tumor_files)
    labels = np.array([1] * len(tumor_files) + [0] * len(no_tumor_files))

    indices = np.arange(len(file_paths))
    np.random.shuffle(indices)

    file_paths, labels = file_paths[indices], labels[indices]

    def load_and_preprocess(file_path, label):
        volume = tf.py_function(func=preprocess_mri_jpg, inp=[file_path], Tout=tf.float32)
        volume.set_shape((128, 128, 64, 1))
        return volume, label

    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# 3D CNN Model
def build_3d_cnn(input_shape=(128, 128, 64, 1)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv3D(32, (3, 3, 3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling3D((2, 2, 2))(x)
    x = tf.keras.layers.Conv3D(64, (3, 3, 3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling3D((2, 2, 2))(x)
    x = tf.keras.layers.Conv3D(128, (3, 3, 3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling3D((2, 2, 2))(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

if __name__ == "__main__":
    tumor_dir = "./MRI_BRAIN/yes"  # Change to actual path
    no_tumor_dir = "./MRI_BRAIN/no"  # Change to actual path
    batch_size = 8

    dataset = create_dataset(tumor_dir, no_tumor_dir, batch_size)
    model = build_3d_cnn()

    model.fit(dataset, epochs=10)
    model.save("brain_tumor_model.h5")



Epoch 1/10


ValueError: Unexpected result of `train_function` (Empty logs). Please use `Model.compile(..., run_eagerly=True)`, or `tf.config.run_functions_eagerly(True)` for more information of where went wrong, or file a issue/bug to `tf.keras`.